# 2. Just enough Python

Module 2 of Etalon's free course on trading bots. You will learn only the Python a bot uses, then load real SPY prices and draw your first chart.

**How to use this notebook**

- Run each cell in order: click it and press **Shift + Enter**, or press the play button on its left.
- If something goes wrong, use **Runtime > Restart session**, then run from the top again.
- Nothing here connects to a broker or touches money.

## 1. Your first cell

Run this one.

In [ ]:
print("hello, market")

The last line of a cell is shown on its own, without `print`:

In [ ]:
2 + 2

## 2. Variables

A variable is a name for a value. Once a value has a name, you can use the name instead.

In [ ]:
price = 200.0
shares = 10
position = price * shares
position

**Try it:** change `shares` to 25 and run the cell again.

## 3. Lists

A list holds several values in order. Here are five made-up closing prices.

In [ ]:
closes = [200.0, 202.5, 201.0, 204.0, 203.0]
print(len(closes), "closes")
print("first:", closes[0])
print("last:", closes[-1])

Python counts from 0, so `closes[0]` is the first one. `closes[-1]` is always the last.

## 4. Loops and `if`

A loop runs the same lines once for each step. An `if` runs lines only when something is true.

In [ ]:
for i in range(1, len(closes)):
    today = closes[i]
    yesterday = closes[i - 1]
    if today > yesterday:
        print(f"day {i}: up, {yesterday} to {today}")
    else:
        print(f"day {i}: not up, {yesterday} to {today}")

The indented lines belong to the loop or to the `if` above them. Python uses the indent for that, so the spaces matter.

This is a bot in miniature: look at each bar, compare it with the one before, and decide.

## 5. Functions

A function is a few lines with a name, so you can reuse them. This one works out a percent change.

In [ ]:
def percent_change(old, new):
    return (new - old) / old * 100

percent_change(200.0, 204.0)

**Try it:** what is the percent change from the first close in `closes` to the last? Use the function.

In [ ]:
# your turn


## 6. Real prices

Two tools from here on:

- **pandas** holds a table, like a spreadsheet with dates down the side.
- **yfinance** fetches prices from Yahoo Finance. It is free and unofficial, and Yahoo's data is for personal use, which learning is.

The first cell makes sure yfinance is up to date. It can take a few seconds.

In [ ]:
%pip install -q --upgrade yfinance

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

Now load every daily bar of SPY since 2015. `auto_adjust=True` means the prices are adjusted for dividends and splits. Module 3 explains what that changes.

In [ ]:
spy = None
try:
    spy = yf.Ticker("SPY").history(start="2015-01-01", auto_adjust=True, actions=False)
    if spy.empty:
        raise ValueError("no rows came back")
    spy.index = spy.index.tz_localize(None)  # daily bars, so the time zone only gets in the way
    print(f"Got {len(spy)} days of SPY, {spy.index[0].date()} to {spy.index[-1].date()}")
except Exception as err:
    spy = None
    print("Yahoo did not answer:", err)
    print("Use Plan B in the next cell.")

### Plan B, only if the cell above failed

Yahoo sometimes turns Colab away. If it did:

1. Open [SPY on Stooq](https://stooq.com/q/d/?s=spy.us) in your browser.
2. Under the table of prices, click the small text link **Download data in csv file...**. Ignore any big "Download" button in an ad: those are adverts, not the data.
3. Run the next cell and pick the file you downloaded.

Stooq adjusts its prices in its own way, so some numbers below will not match Yahoo's exactly. Module 3 covers why.

In [ ]:
if spy is None:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    spy = pd.read_csv(name, index_col="Date", parse_dates=True).loc["2015-01-01":]
    print(f"Got {len(spy)} days of SPY from your file")
else:
    print("Not needed, Yahoo answered.")

## 7. Look at the table

`head()` shows the first rows and `tail()` the last.

In [ ]:
spy.head()

In [ ]:
spy.tail()

Each row is one daily bar from Lesson 1: Open, High, Low, Close and Volume. The dates down the side are the table's **index**.

In [ ]:
print(spy.shape, "rows and columns")
print(list(spy.columns))

## 8. One column, one day

Square brackets pick a column. `.loc` picks a row by its date.

In [ ]:
close = spy["Close"]
close.tail()

In [ ]:
day = spy.loc["2020-03-16"]
day

**Try it:** work out that day's range (High minus Low), and where it closed in its range, as in Lesson 1's task.

In [ ]:
# your turn


## 9. Daily returns

A return is the percent change from one close to the next. pandas does the whole column at once.

In [ ]:
spy["Return"] = spy["Close"].pct_change() * 100
spy["Return"].describe().round(2)

The first day has no return, because there is no day before it. pandas shows that as `NaN`, "not a number".

Now the biggest moves since 2015:

In [ ]:
print("Five worst days, percent:")
print(spy["Return"].nsmallest(5).round(2))
print()
print("Five best days, percent:")
print(spy["Return"].nlargest(5).round(2))

Look at the dates. Most of the worst days and most of the best days fall in the same few weeks. Big moves come in bunches, and a bot that only expects ordinary days is in for a surprise.

## 10. A moving average

A 50-day moving average is the average of the last 50 closes, worked out again every day.

In [ ]:
spy["MA50"] = spy["Close"].rolling(50).mean()
spy[["Close", "MA50"]].iloc[45:55]

The first 49 rows are empty, because the average needs 50 closes before it exists.

Each value only uses closes up to and including its own day. You only know that day's average once the day has closed, so in Module 5 you will act on it the next day, never the same day.

## 11. Draw it

In [ ]:
ax = spy[["Close", "MA50"]].plot(figsize=(11, 4), title="SPY daily close and its 50-day average")
ax.set_ylabel("Adjusted price, USD")
ax.set_xlabel("")
plt.show()

## Your task

1. Add a 200-day average, as a column called `MA200`.
2. Draw the last two years only, with `Close`, `MA50` and `MA200`. Hint: `spy.index[-1]` is the last date, and `pd.DateOffset(years=2)` is two years.
3. Count the days when the close was above its 200-day average, and the share of days that is. Hint: `spy["Close"] > spy["MA200"]` gives True or False for each day, and `.sum()` counts the Trues.

In [ ]:
# your turn


### Answer, try first

In [ ]:
spy["MA200"] = spy["Close"].rolling(200).mean()

recent = spy.loc[spy.index[-1] - pd.DateOffset(years=2):]
ax = recent[["Close", "MA50", "MA200"]].plot(figsize=(11, 4), title="SPY, the last two years")
ax.set_xlabel("")
plt.show()

has_average = spy["MA200"].notna()
above = (spy["Close"] > spy["MA200"])[has_average]
print(f"Above its 200-day average on {above.sum()} of {len(above)} days, {above.mean():.0%}")

Why only the days that have an average: before day 200 there is nothing to compare with, and counting those days as "not above" would be wrong. Small choices like that one change results, and Module 3 is full of them.

## Check yourself

1. The first 49 values of `MA50` are empty. Why, and what would be wrong with filling them in with the first close?
2. The close is above its 200-day average and you want to buy. From what this notebook showed, can you say whether that is a good rule? What would you need to know first?

Nothing in this notebook is a signal. A close above its average describes the past. Whether a rule built on it would have made money after costs is Module 5's question, and whether that would last is Module 6's.

Next: **3. The data.**